## 🧬 NGS Mini‑Pipeline Training

## 🧭 Mapping reads to a reference genome

So far, we have explored what sequencing data looks like. We now have FASTQ files containing short DNA reads.

The next step is to determine **where these reads come from in the genome**.

This process is called **alignment** or **mapping**.

---

### 🧠 What does "mapping" mean?

Each read is a short DNA sequence. On its own, it has no context.

Mapping answers the question:

👉 *Where in the reference genome does this sequence best fit?*

To do this, we compare each read against a **reference genome** (e.g. human genome) and find the best matching location.

---

### 🔬 Analogy

Think of it like this:

- The **reference genome** is a large book  
- Each **read** is a small sentence fragment  
- Mapping is the process of finding **where that sentence belongs in the book**

---

### ⚙️ Tools used in this notebook

| Tool | Purpose |
|---|---|
| **BWA** | Align reads to the reference genome |
| **samtools** | Convert, sort, index, and summarise alignment files |

---

The following command maps our paired-end reads (`sample2_R1.fastq` and `sample2_R2.fastq`) to the mini reference genome we prepared for this training resource.

The output is a **SAM file** (Sequence Alignment/Map) — a text format that records where each read mapped in the genome.

| Part | What it does |
|---|---|
| `bwa mem` | The BWA aligner in paired-end mode |
| `../ref_data/mini_reference.fa` | The reference genome (already indexed) |
| `../fastq_files/sample2_R1.fastq` | Forward reads |
| `../fastq_files/sample2_R2.fastq` | Reverse reads |
| `> ../fastq_files/aligned.sam` | Redirect output to a SAM file |

In [ ]:
!bwa mem ../ref_data/mini_reference.fa ../fastq_files/sample2_R1.fastq ../fastq_files/sample2_R2.fastq > ../fastq_files/aligned.sam

Now let's look at the first few lines of the SAM file to understand its structure.

We use `head -40` to show just 40 lines — enough to see both the header section and a few alignment records.

In [ ]:
!head -40 ../fastq_files/aligned.sam

---

### 📄 Understanding the SAM format

A SAM file has two sections:

**Header lines** — start with `@` and describe the file metadata:

| Tag | Meaning |
|---|---|
| `@HD` | File format version and sort order |
| `@SQ` | Reference sequence name and length |
| `@PG` | Program used to generate the file (BWA in our case) |

**Alignment lines** — one line per read, with 11 mandatory tab-separated fields:

| Field | Name | Example | Meaning |
|---|---|---|---|
| 1 | QNAME | `read1/1` | Read name |
| 2 | FLAG | `99` | Bitwise flag (encodes mapping status, strand, pairing) |
| 3 | RNAME | `chr1` | Reference sequence the read mapped to |
| 4 | POS | `100023` | 1-based leftmost mapping position |
| 5 | MAPQ | `60` | Mapping quality (0–60; higher = more confident) |
| 6 | CIGAR | `150M` | How the read aligns (M=match, I=insertion, D=deletion, S=soft-clip) |
| 7 | RNEXT | `=` | Reference of the mate read (`=` means same chromosome) |
| 8 | PNEXT | `100300` | Position of the mate read |
| 9 | TLEN | `300` | Inferred insert size |
| 10 | SEQ | `ACGT...` | Read sequence |
| 11 | QUAL | `FFFF...` | Base quality scores (same encoding as FASTQ) |

> 💡 The **FLAG** field is a compact way to encode multiple properties in a single number. A FLAG of `99` means: read is paired, properly paired, mate is on the reverse strand, and this is read 1. You can decode any FLAG at [broadinstitute.github.io/picard/explain-flags.html](https://broadinstitute.github.io/picard/explain-flags.html)

---

## 🏥 A real clinical pipeline command

The command we used above is simplified for training purposes. Below is the actual BWA mapping command used in a clinical genotyping pipeline, shown first as-is and then annotated line by line.

```bash
bwa mem -Y -L 50 -K 100000000 -v 3 $bwarefgenomepath FASTQ_R1.fastq FASTQ_R2.fastq | \
java -Djava.io.tmpdir=$PWD -Xmx8G -jar picard.jar \
    MergeBamAlignment \
    REFERENCE_SEQUENCE=$bwarefgenomepath \
    UNMAPPED_BAM=input.bam \
    ALIGNED_BAM=/dev/stdin \
    OUTPUT=sample_FINAL_mapped.bam \
    CREATE_INDEX=true \
    VALIDATION_STRINGENCY=STRICT \
    ATTRIBUTES_TO_RETAIN=X0 \
    SORT_ORDER=coordinate \
    IS_BISULFITE_SEQUENCE=false \
    ALIGNED_READS_ONLY=false \
    CLIP_ADAPTERS=false \
    MAX_RECORDS_IN_RAM=2000000 \
    ADD_MATE_CIGAR=false \
    MAX_INSERTIONS_OR_DELETIONS=-1 \
    PRIMARY_ALIGNMENT_STRATEGY=BestMapq \
    ALIGNER_PROPER_PAIR_FLAGS=false \
    ADD_PG_TAG_TO_READS=false \
    UNMAP_CONTAMINANT_READS=false \
    ATTRIBUTES_TO_RETAIN=X0 \
    ATTRIBUTES_TO_REMOVE=NM \
    ATTRIBUTES_TO_REMOVE=MD
```

Above is how is in the pipeline but I have created a code with comments in between to allow you to understand what is going on

```bash
# -------------------------------
# Step 1: Align reads with BWA
# -------------------------------

bwa mem \
    -Y \                         # Use soft clipping for supplementary alignments (better compatibility with downstream tools)
    -L 50 \                      # Clipping penalty (controls how ends of reads are handled)
    -K 100000000 \               # Process reads in large chunks (performance optimisation)
    -v 3 \                       # Verbosity level (logging detail)
    $bwarefgenomepath \          # Reference genome (must be indexed)
    FASTQ_R1.fastq \             # Forward reads
    FASTQ_R2.fastq \             # Reverse reads

# Pipe output directly to next tool (no intermediate file)
| \

# -------------------------------
# Step 2: Process alignment with Picard
# -------------------------------

java \
    -Djava.io.tmpdir=$PWD \      # Use current directory for temporary files
    -Xmx8G \                     # Allocate 8 GB of RAM
    -jar picard.jar \            # Run Picard tools

    MergeBamAlignment \          # Tool to merge alignment with metadata

    REFERENCE_SEQUENCE=$bwarefgenomepath \   # Reference genome
    UNMAPPED_BAM=input.bam \                 # Original unmapped BAM (contains metadata)
    ALIGNED_BAM=/dev/stdin \                 # Take aligned reads from BWA (via pipe)

    OUTPUT=sample_FINAL_mapped.bam \         # Final output BAM file
    CREATE_INDEX=true \                      # Create BAM index (.bai)

    VALIDATION_STRINGENCY=STRICT \           # Enforce strict format validation
    SORT_ORDER=coordinate \                  # Sort reads by genomic position

    IS_BISULFITE_SEQUENCE=false \            # Not bisulfite sequencing
    ALIGNED_READS_ONLY=false \               # Keep both mapped and unmapped reads
    CLIP_ADAPTERS=false \                    # Do not trim adapters at this stage

    MAX_RECORDS_IN_RAM=2000000 \             # Memory control for processing
    ADD_MATE_CIGAR=false \                   # Do not add mate CIGAR tags

    MAX_INSERTIONS_OR_DELETIONS=-1 \         # No limit on indels
    PRIMARY_ALIGNMENT_STRATEGY=BestMapq \    # Choose best alignment by mapping quality

    ALIGNER_PROPER_PAIR_FLAGS=false \        # Do not rely on aligner pairing flags
    ADD_PG_TAG_TO_READS=false \              # Do not add program group tags

    UNMAP_CONTAMINANT_READS=false \          # Do not unmap suspected contaminants

    ATTRIBUTES_TO_RETAIN=X0 \               # Keep specific tag (aligner-specific)
    ATTRIBUTES_TO_REMOVE=NM \               # Remove edit distance tag
    ATTRIBUTES_TO_REMOVE=MD                 # Remove mismatch string tag
```

---

## 🔧 Working with SAM/BAM files using samtools

The SAM file produced by BWA is a text file — useful to inspect, but too large and slow to work with directly in most downstream tools.

In practice, we always:
1. **Convert** SAM → BAM (binary, compressed)
2. **Sort** the BAM by genomic coordinate
3. **Index** the sorted BAM (creates a `.bai` file for fast random access)

`samtools` is the standard tool for all of these operations.

### Step 1 — Convert SAM to sorted BAM

`samtools sort` converts the SAM to BAM and sorts the reads by coordinate in one step.  
The `-o` flag specifies the output file.

In [ ]:
!samtools sort ../fastq_files/aligned.sam -o ../fastq_files/aligned_sorted.bam

### Step 2 — Index the BAM file

Indexing creates a `.bai` file alongside the BAM. This allows tools to jump directly to any genomic region without reading the whole file — essential for variant calling and visualisation.

In [ ]:
!samtools index ../fastq_files/aligned_sorted.bam

### Step 3 — Check mapping statistics with flagstat

`samtools flagstat` reads the FLAG field of every alignment and produces a summary of the mapping results.

In [ ]:
!samtools flagstat ../fastq_files/aligned_sorted.bam

The output will show you how many reads:
- were processed in total
- mapped successfully
- were properly paired
- were duplicates or secondary alignments

> 👉 What percentage of reads mapped? Is this what you would expect for a small synthetic dataset?

---

## 🖥️ Visualising the alignment with IGV

Numbers and text are useful, but the best way to understand an alignment is to **look at it**.

**IGV (Integrative Genomics Viewer)** is a tool widely used in clinical and research settings to visualise reads aligned to a reference genome. Here we use **igv-notebook**, which embeds IGV directly inside this notebook — no installation or external browser required.

First, we need to index the reference FASTA file so IGV can access it efficiently:

In [ ]:
!samtools faidx ../ref_data/mini_reference.fa

Now launch the IGV browser. It will load our mini reference and the sorted BAM file we just created:

In [ ]:
import igv_notebook
igv_notebook.init()

b = igv_notebook.Browser({
    "reference": {
        "id": "mini_reference",
        "fastaURL": "../ref_data/mini_reference.fa",
        "indexURL": "../ref_data/mini_reference.fa.fai"
    },
    "locus": "chr1:100000-101000"
})

b.load_track({
    "name": "Aligned reads",
    "url": "../fastq_files/aligned_sorted.bam",
    "indexURL": "../fastq_files/aligned_sorted.bam.bai",
    "format": "bam",
    "type": "alignment"
})

Each horizontal bar represents one read. The colours indicate:
- **Grey bars** — reads that map to this region
- **Coloured bases** — positions where a read differs from the reference
- **Arrows on reads** — indicate the strand direction (forward or reverse)

> 👉 Can you spot any positions where most reads show the same base change? That could be a variant.  
> 👉 Zoom in to position `chr1:100200` — do you notice anything unusual?

This is exactly how clinical bioinformaticians visually inspect alignments to validate variant calls before reporting.

## About the Author

<p align="center">
  <img src="https://raw.githubusercontent.com/Manuel-DominguezCBG/ngs-teaching-binder/main/assets/Manolo.jpg" width="120" style="border-radius: 6px;">
</p>

**Manuel — Bioinformatician**

Manuel works as a bioinformatician in clinical genomics. He created this resource to support trainees taking their first steps in NGS bioinformatics, with the aim of making the learning process a little less daunting.

Feedback, adaptations, and contributions to this teaching resource are welcome.

**LinkedIn:** [Manuel J. Domínguez](https://www.linkedin.com/in/manuel-j-dom%C3%ADnguez-aa97981b2/)  
**GitHub Repository:** [ngs‑teaching‑binder](https://github.com/Manuel-DominguezCBG/ngs-teaching-binder)

**Huge thanks to the Bioinformatics team and all my WGLS colleagues for helping me make these learning resources better.**